# OptiForge3D API Server with Cloudflare Tunnel
This notebook installs dependencies, uploads your local scripts, uses `download_models.py` to directly fetch the models (bypassing Google Drive), and exposes a memory-optimized API.


## 1. Install Dependencies
Installing libraries required by OptiForge3D: `shap-e` (from GitHub), `ultralytics` for YOLOv8, `transformers` for CLIP, plus FastAPI and Cloudflare components.


In [1]:
!pip install -q fastapi uvicorn nest-asyncio pydantic torch torchvision transformers ultralytics "git+https://github.com/openai/shap-e.git" pyngrok


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


## 2. Create Project Scripts
This cell automatically creates the necessary python files on the Colab disk.


In [2]:
with open("download_models.py", "wb") as f:
    f.write(__import__("base64").b64decode("IiIiCmRvd25sb2FkX21vZGVscy5weQotLS0tLS0tLS0tLS0tLS0tLS0tClJ1biB0aGlzIG9uY2UgKHBlciBtYWNoaW5lKSB0byBwdWxsIFNoYXAtRSwgWU9MT3Y4LCBhbmQgQ0xJUCB3ZWlnaHRzIGludG8KYSBsb2NhbCBgbW9kZWxzL2AgZGlyZWN0b3J5LiBUaGlzIGRpcmVjdG9yeSBtdXN0IGJlIGdpdGlnbm9yZWQg4oCUIHNlZQpgLmdpdGlnbm9yZS5zbmlwcGV0YCBpbiB0aGlzIGZvbGRlci4KClVzYWdlOgogICAgcHl0aG9uIGRvd25sb2FkX21vZGVscy5weSAgICAgICAgICAgICMgZG93bmxvYWRzIGFsbCB0aHJlZQogICAgcHl0aG9uIGRvd25sb2FkX21vZGVscy5weSAtLW9ubHkgeW9sbwogICAgcHl0aG9uIGRvd25sb2FkX21vZGVscy5weSAtLW9ubHkgY2xpcAogICAgcHl0aG9uIGRvd25sb2FkX21vZGVscy5weSAtLW9ubHkgc2hhcGUKCldlaWdodHMgbGFuZCBpbjoKICAgIG1vZGVscy8KICAgICAgc2hhcC1lLwogICAgICB5b2xvdjgvCiAgICAgICAgeW9sb3Y4bi5wdCAgIChvciB3aGljaGV2ZXIgdmFyaWFudCB5b3UgY29uZmlndXJlIGJlbG93KQogICAgICBjbGlwLwoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyBbJShsZXZlbG5hbWUpc10gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigib3B0aWZvcmdlM2QuZG93bmxvYWQiKQoKTU9ERUxTX0RJUiA9IFBhdGgob3MuZ2V0ZW52KCJPUFRJRk9SR0VfTU9ERUxTX0RJUiIsICIuL21vZGVscyIpKS5yZXNvbHZlKCkKCiMgUGljayB0aGUgWU9MT3Y4IHZhcmlhbnQgeW91IGFjdHVhbGx5IG5lZWQgKG4gPSBuYW5vLCBmYXN0ZXN0L3NtYWxsZXN0OwojIHMvbS9sL3ggPSBwcm9ncmVzc2l2ZWx5IGxhcmdlciArIG1vcmUgYWNjdXJhdGUpLiBGb3IgcHJlcHJvY2Vzc2luZy8KIyBvYmplY3QgY3JvcHBpbmcgaW4gT3B0aUZvcmdlM0QsIGB5b2xvdjhuLnB0YCBvciBgeW9sb3Y4cy5wdGAgaXMgdXN1YWxseSBlbm91Z2guCllPTE9fVkFSSUFOVCA9IG9zLmdldGVudigiT1BUSUZPUkdFX1lPTE9fVkFSSUFOVCIsICJ5b2xvdjhuLnB0IikKCkNMSVBfTU9ERUxfTkFNRSA9IG9zLmdldGVudigiT1BUSUZPUkdFX0NMSVBfTU9ERUwiLCAib3BlbmFpL2NsaXAtdml0LWJhc2UtcGF0Y2gzMiIpCgoKZGVmIGRvd25sb2FkX3NoYXBfZSgpIC0+IE5vbmU6CiAgICAiIiIKICAgIFNoYXAtRSBzaGlwcyB0aHJvdWdoIHRoZSBgc2hhcC1lYCBwYWNrYWdlIChPcGVuQUkpLiBJdHMgYGxvYWRfbW9kZWxgCiAgICBoZWxwZXIgZG93bmxvYWRzIHRvIGl0cyBvd24gY2FjaGUgZGlyIGJ5IGRlZmF1bHQ7IGhlcmUgd2UgcmVkaXJlY3QKICAgIHRoYXQgY2FjaGUgaW50byBvdXIgbG9jYWwsIGdpdGlnbm9yZWQgbW9kZWxzLyBmb2xkZXIgc28gZXZlcnl0aGluZwogICAgbGl2ZXMgaW4gb25lIHByZWRpY3RhYmxlIHBsYWNlLgogICAgIiIiCiAgICB0YXJnZXQgPSBNT0RFTFNfRElSIC8gInNoYXAtZSIKICAgIHRhcmdldC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyBzaGFwLWUgcmVhZHMgdGhpcyBlbnYgdmFyIGZvciB3aGVyZSB0byBjYWNoZSBjaGVja3BvaW50cwogICAgb3MuZW52aXJvblsiWERHX0NBQ0hFX0hPTUUiXSA9IHN0cih0YXJnZXQpCgogICAgdHJ5OgogICAgICAgIGZyb20gc2hhcF9lLm1vZGVscy5kb3dubG9hZCBpbXBvcnQgbG9hZF9tb2RlbAogICAgICAgIGZyb20gc2hhcF9lLmRpZmZ1c2lvbi5zYW1wbGUgaW1wb3J0IHNhbXBsZV9sYXRlbnRzICAjIG5vcWE6IEY0MDEgKGltcG9ydCBjaGVjaykKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBsb2dnZXIuZXJyb3IoCiAgICAgICAgICAgICJzaGFwLWUgaXMgbm90IGluc3RhbGxlZC4gSW5zdGFsbCBpdCBmaXJzdDpcbiIKICAgICAgICAgICAgIiAgcGlwIGluc3RhbGwgZ2l0K2h0dHBzOi8vZ2l0aHViLmNvbS9vcGVuYWkvc2hhcC1lLmdpdCIKICAgICAgICApCiAgICAgICAgcmV0dXJuCgogICAgbG9nZ2VyLmluZm8oIkRvd25sb2FkaW5nIFNoYXAtRSBjaGVja3BvaW50cyAodGV4dDMwME0sIGltYWdlMzAwTSwgdHJhbnNtaXR0ZXIpLi4uIikKICAgIGZvciBuYW1lIGluIFsidGV4dDMwME0iLCAiaW1hZ2UzMDBNIiwgInRyYW5zbWl0dGVyIl06CiAgICAgICAgbG9nZ2VyLmluZm8oIiAgLT4gJXMiLCBuYW1lKQogICAgICAgIGxvYWRfbW9kZWwobmFtZSwgZGV2aWNlPSJjcHUiKSAgIyBkZXZpY2UgZG9lc24ndCBtYXR0ZXIgZm9yIHRoZSBkb3dubG9hZCBzdGVwCgogICAgbG9nZ2VyLmluZm8oIlNoYXAtRSB3ZWlnaHRzIGNhY2hlZCB1bmRlcjogJXMiLCB0YXJnZXQpCgoKZGVmIGRvd25sb2FkX3lvbG92OCgpIC0+IE5vbmU6CiAgICB0YXJnZXQgPSBNT0RFTFNfRElSIC8gInlvbG92OCIKICAgIHRhcmdldC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgdHJ5OgogICAgICAgIGZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBsb2dnZXIuZXJyb3IoInVsdHJhbHl0aWNzIGlzIG5vdCBpbnN0YWxsZWQuIEluc3RhbGwgaXQgZmlyc3Q6XG4gIHBpcCBpbnN0YWxsIHVsdHJhbHl0aWNzIikKICAgICAgICByZXR1cm4KCiAgICBkZXN0X3BhdGggPSB0YXJnZXQgLyBZT0xPX1ZBUklBTlQKICAgIGlmIGRlc3RfcGF0aC5leGlzdHMoKToKICAgICAgICBsb2dnZXIuaW5mbygiWU9MT3Y4IHdlaWdodHMgYWxyZWFkeSBwcmVzZW50IGF0ICVzLCBza2lwcGluZy4iLCBkZXN0X3BhdGgpCiAgICAgICAgcmV0dXJuCgogICAgbG9nZ2VyLmluZm8oIkRvd25sb2FkaW5nIFlPTE92OCB3ZWlnaHRzOiAlcyIsIFlPTE9fVkFSSUFOVCkKICAgICMgSW5zdGFudGlhdGluZyBZT0xPKHZhcmlhbnQpIGF1dG8tZG93bmxvYWRzIGludG8gdGhlIGN1cnJlbnQgd29ya2luZyBkaXIKICAgIG9yaWdpbmFsX2N3ZCA9IG9zLmdldGN3ZCgpCiAgICB0cnk6CiAgICAgICAgb3MuY2hkaXIodGFyZ2V0KQogICAgICAgIFlPTE8oWU9MT19WQVJJQU5UKSAgIyB0cmlnZ2VycyB0aGUgZG93bmxvYWQKICAgIGZpbmFsbHk6CiAgICAgICAgb3MuY2hkaXIob3JpZ2luYWxfY3dkKQoKICAgIGxvZ2dlci5pbmZvKCJZT0xPdjggd2VpZ2h0cyBzYXZlZCB1bmRlcjogJXMiLCB0YXJnZXQpCgoKZGVmIGRvd25sb2FkX2NsaXAoKSAtPiBOb25lOgogICAgdGFyZ2V0ID0gTU9ERUxTX0RJUiAvICJjbGlwIgogICAgdGFyZ2V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICB0cnk6CiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IENMSVBNb2RlbCwgQ0xJUFByb2Nlc3NvcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIGxvZ2dlci5lcnJvcigidHJhbnNmb3JtZXJzIGlzIG5vdCBpbnN0YWxsZWQuIEluc3RhbGwgaXQgZmlyc3Q6XG4gIHBpcCBpbnN0YWxsIHRyYW5zZm9ybWVycyIpCiAgICAgICAgcmV0dXJuCgogICAgbG9nZ2VyLmluZm8oIkRvd25sb2FkaW5nIENMSVAgd2VpZ2h0czogJXMiLCBDTElQX01PREVMX05BTUUpCiAgICBtb2RlbCA9IENMSVBNb2RlbC5mcm9tX3ByZXRyYWluZWQoQ0xJUF9NT0RFTF9OQU1FLCBjYWNoZV9kaXI9c3RyKHRhcmdldCkpCiAgICBwcm9jZXNzb3IgPSBDTElQUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZChDTElQX01PREVMX05BTUUsIGNhY2hlX2Rpcj1zdHIodGFyZ2V0KSkKICAgIGRlbCBtb2RlbCwgcHJvY2Vzc29yCgogICAgbG9nZ2VyLmluZm8oIkNMSVAgd2VpZ2h0cyBjYWNoZWQgdW5kZXI6ICVzIiwgdGFyZ2V0KQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iRG93bmxvYWQgT3B0aUZvcmdlM0QgbW9kZWwgd2VpZ2h0cyBsb2NhbGx5LiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLW9ubHkiLAogICAgICAgIGNob2ljZXM9WyJzaGFwZSIsICJ5b2xvIiwgImNsaXAiXSwKICAgICAgICBoZWxwPSJEb3dubG9hZCBvbmx5IG9uZSBtb2RlbCBpbnN0ZWFkIG9mIGFsbCB0aHJlZS4iLAogICAgKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBNT0RFTFNfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGxvZ2dlci5pbmZvKCJNb2RlbHMgd2lsbCBiZSBzdG9yZWQgdW5kZXI6ICVzIiwgTU9ERUxTX0RJUikKCiAgICB0YXNrcyA9IHsKICAgICAgICAic2hhcGUiOiBkb3dubG9hZF9zaGFwX2UsCiAgICAgICAgInlvbG8iOiBkb3dubG9hZF95b2xvdjgsCiAgICAgICAgImNsaXAiOiBkb3dubG9hZF9jbGlwLAogICAgfQoKICAgIGlmIGFyZ3Mub25seToKICAgICAgICB0YXNrc1thcmdzLm9ubHldKCkKICAgIGVsc2U6CiAgICAgICAgZm9yIGZuIGluIHRhc2tzLnZhbHVlcygpOgogICAgICAgICAgICBmbigpCgogICAgbG9nZ2VyLmluZm8oIkRvbmUuIFJlbWluZGVyOiBtb2RlbHMvIG11c3Qgc3RheSBvdXQgb2YgR2l0IOKAlCBjaGVjayAuZ2l0aWdub3JlLiIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="))

with open("loaders.py", "wb") as f:
    f.write(__import__("base64").b64decode("IiIiCmxvYWRlcnMucHkKLS0tLS0tLS0tLQpUaGluLCBtZW1vcnktYXdhcmUgbG9hZGVyIGZ1bmN0aW9ucyBmb3IgZWFjaCBtb2RlbC4gVGhlc2UgYXJlIHdoYXQgdGhlCkNlbGVyeSB0YXNrcyAvIEZhc3RBUEkgd29ya2VycyBzaG91bGQgaW1wb3J0IOKAlCBuZXZlciBsb2FkIGEgbW9kZWwgd2l0aAphZC1ob2MgY29kZSBzY2F0dGVyZWQgYWNyb3NzIHRoZSBwaXBlbGluZS4KCkVhY2ggbG9hZGVyOgogIDEuIEFzc3VtZXMgd2VpZ2h0cyB3ZXJlIGFscmVhZHkgZmV0Y2hlZCB2aWEgZG93bmxvYWRfbW9kZWxzLnB5CiAgMi4gVXNlcyBncHVfY29uZmlnLmdldF9kZXZpY2UoKSBzbyBkZXZpY2Ugc2VsZWN0aW9uIHN0YXlzIGluIG9uZSBwbGFjZQogIDMuIFB1dHMgdGhlIG1vZGVsIGluIGV2YWwoKSBtb2RlIGFuZCBkaXNhYmxlcyBncmFkIHRyYWNraW5nIHdoZXJlIHJlbGV2YW50CiIiIgoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKCmZyb20gZ3B1X2NvbmZpZyBpbXBvcnQgZ2V0X2RldmljZSwgTUFYX0dQVV9NRU1PUllfRlJBQ1RJT04gICMgbm9xYTogRjQwMQoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIm9wdGlmb3JnZTNkLmxvYWRlcnMiKQoKTU9ERUxTX0RJUiA9IFBhdGgoIi4vbW9kZWxzIikucmVzb2x2ZSgpCgoKZGVmIGxvYWRfc2hhcF9lKG1vZGVsX25hbWU6IHN0ciA9ICJ0ZXh0MzAwTSIpOgogICAgIiIiCiAgICBtb2RlbF9uYW1lOiAidGV4dDMwME0iICh0ZXh0LXRvLTNEKSwgImltYWdlMzAwTSIgKGltYWdlLXRvLTNEKSwKICAgICAgICAgICAgICAgIG9yICJ0cmFuc21pdHRlciIgKGxhdGVudCBkZWNvZGVyKS4KICAgICIiIgogICAgZnJvbSBzaGFwX2UubW9kZWxzLmRvd25sb2FkIGltcG9ydCBsb2FkX21vZGVsCiAgICBmcm9tIHNoYXBfZS51dGlsLm5vdGVib29rcyBpbXBvcnQgZGVjb2RlX2xhdGVudF9tZXNoICAjIG5vcWE6IEY0MDEgKHVzZWQgZG93bnN0cmVhbSkKCiAgICBkZXZpY2UgPSBnZXRfZGV2aWNlKCkKICAgIGxvZ2dlci5pbmZvKCJMb2FkaW5nIFNoYXAtRSBtb2RlbCAnJXMnIG9uICVzIiwgbW9kZWxfbmFtZSwgZGV2aWNlKQoKICAgIG1vZGVsID0gbG9hZF9tb2RlbChtb2RlbF9uYW1lLCBkZXZpY2U9ZGV2aWNlKQogICAgaWYgaGFzYXR0cihtb2RlbCwgImV2YWwiKToKICAgICAgICBtb2RlbC5ldmFsKCkKICAgIHJldHVybiBtb2RlbCwgZGV2aWNlCgoKZGVmIGxvYWRfeW9sb3Y4KHZhcmlhbnQ6IHN0ciA9ICJ5b2xvdjhuLnB0Iik6CiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCgogICAgZGV2aWNlID0gZ2V0X2RldmljZSgpCiAgICB3ZWlnaHRzX3BhdGggPSBNT0RFTFNfRElSIC8gInlvbG92OCIgLyB2YXJpYW50CiAgICBpZiBub3Qgd2VpZ2h0c19wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIllPTE92OCB3ZWlnaHRzIG5vdCBmb3VuZCBhdCB7d2VpZ2h0c19wYXRofS4gIgogICAgICAgICAgICBmIlJ1biBgcHl0aG9uIGRvd25sb2FkX21vZGVscy5weSAtLW9ubHkgeW9sb2AgZmlyc3QuIgogICAgICAgICkKCiAgICBsb2dnZXIuaW5mbygiTG9hZGluZyBZT0xPdjggKCVzKSBvbiAlcyIsIHZhcmlhbnQsIGRldmljZSkKICAgIG1vZGVsID0gWU9MTyhzdHIod2VpZ2h0c19wYXRoKSkKICAgIG1vZGVsLnRvKGRldmljZSkKICAgIHJldHVybiBtb2RlbAoKCmRlZiBsb2FkX2NsaXAobW9kZWxfbmFtZTogc3RyID0gIm9wZW5haS9jbGlwLXZpdC1iYXNlLXBhdGNoMzIiKToKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBDTElQTW9kZWwsIENMSVBQcm9jZXNzb3IKCiAgICBkZXZpY2UgPSBnZXRfZGV2aWNlKCkKICAgIGNhY2hlX2RpciA9IE1PREVMU19ESVIgLyAiY2xpcCIKICAgIGlmIG5vdCBjYWNoZV9kaXIuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiQ0xJUCBjYWNoZSBub3QgZm91bmQgYXQge2NhY2hlX2Rpcn0uICIKICAgICAgICAgICAgZiJSdW4gYHB5dGhvbiBkb3dubG9hZF9tb2RlbHMucHkgLS1vbmx5IGNsaXBgIGZpcnN0LiIKICAgICAgICApCgogICAgbG9nZ2VyLmluZm8oIkxvYWRpbmcgQ0xJUCAoJXMpIG9uICVzIiwgbW9kZWxfbmFtZSwgZGV2aWNlKQogICAgbW9kZWwgPSBDTElQTW9kZWwuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUsIGNhY2hlX2Rpcj1zdHIoY2FjaGVfZGlyKSkudG8oZGV2aWNlKQogICAgcHJvY2Vzc29yID0gQ0xJUFByb2Nlc3Nvci5mcm9tX3ByZXRyYWluZWQobW9kZWxfbmFtZSwgY2FjaGVfZGlyPXN0cihjYWNoZV9kaXIpKQogICAgbW9kZWwuZXZhbCgpCiAgICByZXR1cm4gbW9kZWwsIHByb2Nlc3NvciwgZGV2aWNlCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBFeGFtcGxlOiBob3cgYSBDZWxlcnkgdGFzayBpbiBPcHRpRm9yZ2UzRCBtaWdodCB1c2UgdGhlc2UgdG9nZXRoZXIKIyB3aXRob3V0IGxldHRpbmcgR1BVIG1lbW9yeSBwaWxlIHVwIGFjcm9zcyBhIGxvbmctcnVubmluZyB3b3JrZXIuCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBleGFtcGxlX3BpcGVsaW5lX3N0ZXAocHJvbXB0OiBzdHIpOgogICAgZnJvbSBncHVfY29uZmlnIGltcG9ydCBjb25maWd1cmVfdG9yY2gsIGZyZWVfZ3B1X21lbW9yeSwgbG9nX2dwdV9tZW1vcnkKCiAgICBjb25maWd1cmVfdG9yY2goKSAgIyBvbmNlIHBlciB3b3JrZXIgcHJvY2Vzcywgc2FmZSB0byBjYWxsIHJlcGVhdGVkbHkKCiAgICBjbGlwX21vZGVsLCBjbGlwX3Byb2Nlc3NvciwgZGV2aWNlID0gbG9hZF9jbGlwKCkKICAgIGxvZ19ncHVfbWVtb3J5KCJhZnRlciBDTElQIGxvYWQiKQoKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGlucHV0cyA9IGNsaXBfcHJvY2Vzc29yKHRleHQ9W3Byb21wdF0sIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKGRldmljZSkKICAgICAgICBfID0gY2xpcF9tb2RlbC5nZXRfdGV4dF9mZWF0dXJlcygqKmlucHV0cykKCiAgICAjIFJlbGVhc2UgQ0xJUCBiZWZvcmUgbG9hZGluZyBTaGFwLUUgaW4gdGhlIHNhbWUgd29ya2VyCiAgICBkZWwgY2xpcF9tb2RlbCwgY2xpcF9wcm9jZXNzb3IKICAgIGZyZWVfZ3B1X21lbW9yeSgpCiAgICBsb2dfZ3B1X21lbW9yeSgiYWZ0ZXIgQ0xJUCByZWxlYXNlIikKCiAgICBzaGFwX2VfbW9kZWwsIGRldmljZSA9IGxvYWRfc2hhcF9lKCJ0ZXh0MzAwTSIpCiAgICBsb2dfZ3B1X21lbW9yeSgiYWZ0ZXIgU2hhcC1FIGxvYWQiKQoKICAgICMgLi4uIHJ1biBnZW5lcmF0aW9uIC4uLgoKICAgIGRlbCBzaGFwX2VfbW9kZWwKICAgIGZyZWVfZ3B1X21lbW9yeSgpCiAgICBsb2dfZ3B1X21lbW9yeSgiYWZ0ZXIgU2hhcC1FIHJlbGVhc2UiKQo="))

with open("gpu_config.py", "wb") as f:
    f.write(__import__("base64").b64decode("IiIiCmdwdV9jb25maWcucHkKLS0tLS0tLS0tLS0tLQpDZW50cmFsIHBsYWNlIGZvciBjb25maWd1cmluZyBQeVRvcmNoIHRvIHVzZSB0aGUgbG9jYWwgTlZJRElBIEdQVSB3aXRoCnN0cmljdCBtZW1vcnkgbWFuYWdlbWVudC4gSW1wb3J0IGBnZXRfZGV2aWNlKClgIGFuZCBgY29uZmlndXJlX3RvcmNoKClgCmZyb20gZXZlcnkgbG9hZGVyIHNjcmlwdCAoU2hhcC1FLCBZT0xPdjgsIENMSVApIGluc3RlYWQgb2YgaGFyZGNvZGluZwpgdG9yY2guZGV2aWNlKC4uLilgIGV2ZXJ5d2hlcmUuCgpXaHkgdGhpcyBtYXR0ZXJzIGZvciBPcHRpRm9yZ2UzRDoKU2hhcC1FLCBZT0xPdjgsIGFuZCBDTElQIHdpbGwgYWxsIGJlIGxvYWRlZCBpbiB0aGUgc2FtZSB3b3JrZXIgcHJvY2VzcwooQ2VsZXJ5IHRhc2spIGF0IHZhcmlvdXMgcG9pbnRzIGluIHRoZSBnZW5lcmF0aW9uIHBpcGVsaW5lLiBXaXRob3V0CmxpbWl0cywgUHlUb3JjaCdzIGNhY2hpbmcgYWxsb2NhdG9yIHdpbGwgaGFwcGlseSBncmFiIG1vc3Qgb2YgdGhlIEdQVQpmb3Igb25lIG1vZGVsIGFuZCBzdGFydmUgdGhlIG90aGVycywgb3IgZnJhZ21lbnQgbWVtb3J5IGJhZGx5IHVuZGVyCkNlbGVyeSdzIGxvbmctcnVubmluZyB3b3JrZXIgcHJvY2Vzc2VzLgoiIiIKCmltcG9ydCBvcwppbXBvcnQgZ2MKaW1wb3J0IGxvZ2dpbmcKCmltcG9ydCB0b3JjaAoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIm9wdGlmb3JnZTNkLmdwdSIpCgojIC0tLS0gVHVuYWJsZXMgKG92ZXJyaWRlIHZpYSBlbnZpcm9ubWVudCB2YXJpYWJsZXMgaW4gZG9ja2VyLWNvbXBvc2UpIC0tLS0KIyBGcmFjdGlvbiBvZiB0b3RhbCBHUFUgbWVtb3J5IHRoaXMgcHJvY2VzcyBpcyBhbGxvd2VkIHRvIHVzZS4KTUFYX0dQVV9NRU1PUllfRlJBQ1RJT04gPSBmbG9hdChvcy5nZXRlbnYoIk9QVElGT1JHRV9NQVhfR1BVX01FTV9GUkFDVElPTiIsICIwLjg1IikpCiMgV2hldGhlciB0byB1c2UgUHlUb3JjaCdzIGV4cGFuZGFibGUgbWVtb3J5IHNlZ21lbnRzIChoZWxwcyBmcmFnbWVudGF0aW9uKS4KVVNFX0VYUEFOREFCTEVfU0VHTUVOVFMgPSBvcy5nZXRlbnYoIk9QVElGT1JHRV9FWFBBTkRBQkxFX1NFR01FTlRTIiwgIjEiKSA9PSAiMSIKIyBGb3JjZSBDUFUgZXZlbiBpZiBhIEdQVSBpcyBwcmVzZW50ICh1c2VmdWwgZm9yIGxvY2FsIGRldiBvbiBhIGxhcHRvcCkuCkZPUkNFX0NQVSA9IG9zLmdldGVudigiT1BUSUZPUkdFX0ZPUkNFX0NQVSIsICIwIikgPT0gIjEiCgoKZGVmIGNvbmZpZ3VyZV90b3JjaCgpIC0+IE5vbmU6CiAgICAiIiIKICAgIENhbGwgdGhpcyBPTkNFIGF0IHByb2Nlc3Mvd29ya2VyIHN0YXJ0dXAgKGUuZy4gaW4gdGhlIENlbGVyeSB3b3JrZXIncwogICAgYHdvcmtlcl9wcm9jZXNzX2luaXRgIHNpZ25hbCwgb3IgYXQgdGhlIHRvcCBvZiBhIHN0YW5kYWxvbmUgc2NyaXB0KQogICAgYmVmb3JlIGFueSBtb2RlbCBpcyBsb2FkZWQuCiAgICAiIiIKICAgIGlmIFVTRV9FWFBBTkRBQkxFX1NFR01FTlRTOgogICAgICAgICMgUmVkdWNlcyBDVURBIG1lbW9yeSBmcmFnbWVudGF0aW9uIGFjcm9zcyByZXBlYXRlZAogICAgICAgICMgYWxsb2NhdGUvZnJlZSBjeWNsZXMsIHdoaWNoIG1hdHRlcnMgYSBsb3Qgd2hlbiB0aGUgc2FtZSB3b3JrZXIKICAgICAgICAjIGxvYWRzL3VubG9hZHMgU2hhcC1FLCBZT0xPdjgsIGFuZCBDTElQIGJhY2stdG8tYmFjay4KICAgICAgICBvcy5lbnZpcm9uLnNldGRlZmF1bHQoCiAgICAgICAgICAgICJQWVRPUkNIX0NVREFfQUxMT0NfQ09ORiIsICJleHBhbmRhYmxlX3NlZ21lbnRzOlRydWUiCiAgICAgICAgKQoKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIGxvZ2dlci53YXJuaW5nKCJDVURBIG5vdCBhdmFpbGFibGUg4oCUIGZhbGxpbmcgYmFjayB0byBDUFUuIikKICAgICAgICByZXR1cm4KCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBUcnVlCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyID0gVHJ1ZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYWxsb3dfdGYzMiA9IFRydWUKCiAgICB0cnk6CiAgICAgICAgdG9yY2guY3VkYS5zZXRfcGVyX3Byb2Nlc3NfbWVtb3J5X2ZyYWN0aW9uKE1BWF9HUFVfTUVNT1JZX0ZSQUNUSU9OLCBkZXZpY2U9MCkKICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgIkNVREEgY29uZmlndXJlZDogZGV2aWNlPSVzLCBtZW1fZnJhY3Rpb249JS4yZiwgZXhwYW5kYWJsZV9zZWdtZW50cz0lcyIsCiAgICAgICAgICAgIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLAogICAgICAgICAgICBNQVhfR1BVX01FTU9SWV9GUkFDVElPTiwKICAgICAgICAgICAgVVNFX0VYUEFOREFCTEVfU0VHTUVOVFMsCiAgICAgICAgKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBzb21lIGRyaXZlci9DVURBIGNvbWJvcyBkb24ndCBzdXBwb3J0IHRoaXMgY2FsbAogICAgICAgIGxvZ2dlci53YXJuaW5nKCJDb3VsZCBub3Qgc2V0IHBlci1wcm9jZXNzIG1lbW9yeSBmcmFjdGlvbjogJXMiLCBlKQoKCmRlZiBnZXRfZGV2aWNlKCkgLT4gdG9yY2guZGV2aWNlOgogICAgIiIiUmV0dXJucyB0aGUgZGV2aWNlIGV2ZXJ5IGxvYWRlciBzaG91bGQgdXNlLiIiIgogICAgaWYgRk9SQ0VfQ1BVIG9yIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImNwdSIpCiAgICByZXR1cm4gdG9yY2guZGV2aWNlKCJjdWRhOjAiKQoKCmRlZiBmcmVlX2dwdV9tZW1vcnkoKSAtPiBOb25lOgogICAgIiIiCiAgICBDYWxsIGFmdGVyIGEgaGVhdnkgaW5mZXJlbmNlIHN0ZXAgKGUuZy4gYWZ0ZXIgYSBTaGFwLUUgZ2VuZXJhdGlvbiBqb2IKICAgIGZpbmlzaGVzKSB0byByZWxlYXNlIGNhY2hlZCBtZW1vcnkgYmFjayB0byB0aGUgYWxsb2NhdG9yIHBvb2wgYmVmb3JlCiAgICB0aGUgbmV4dCBtb2RlbCBydW5zIGluIHRoZSBzYW1lIHdvcmtlci4KICAgICIiIgogICAgZ2MuY29sbGVjdCgpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgICAgIHRvcmNoLmN1ZGEuaXBjX2NvbGxlY3QoKQoKCmRlZiBsb2dfZ3B1X21lbW9yeSh0YWc6IHN0ciA9ICIiKSAtPiBOb25lOgogICAgIiIiTGlnaHR3ZWlnaHQgbWVtb3J5IGxvZ2dpbmcgeW91IGNhbiBzcHJpbmtsZSBhcm91bmQgdGhlIHBpcGVsaW5lLiIiIgogICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcmV0dXJuCiAgICBhbGxvY2F0ZWQgPSB0b3JjaC5jdWRhLm1lbW9yeV9hbGxvY2F0ZWQoKSAvIDEwMjQqKjMKICAgIHJlc2VydmVkID0gdG9yY2guY3VkYS5tZW1vcnlfcmVzZXJ2ZWQoKSAvIDEwMjQqKjMKICAgIGxvZ2dlci5pbmZvKCJbJXNdIEdQVSBtZW0g4oCUIGFsbG9jYXRlZDogJS4yZiBHQiwgcmVzZXJ2ZWQ6ICUuMmYgR0IiLCB0YWcsIGFsbG9jYXRlZCwgcmVzZXJ2ZWQpCg=="))


## 3. Download Models Directly
Run your `download_models.py` script to fetch CLIP, YOLOv8, and SHAP-E weights directly into the `models/` directory.


In [3]:
!python download_models.py


2026-07-02 23:12:04,478 [INFO] Models will be stored under: /content/models
2026-07-02 23:12:11,224 [INFO] Downloading Shap-E checkpoints (text300M, image300M, transmitter)...
2026-07-02 23:12:11,224 [INFO]   -> text300M
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:31: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:43: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:61: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:86: FutureWarning: `torc

## 4. Create the API (FastAPI) with Memory Optimization
This uses `gpu_config.py` and `loaders.py` exactly as intended by your architecture. It initializes torch correctly to prevent fragmentation and frees GPU memory between model loads.


In [4]:
api_script = """
from fastapi import FastAPI, BackgroundTasks
from fastapi.responses import FileResponse
from pydantic import BaseModel
import os
import torch

# Import your OptiForge3D modules
import gpu_config
import loaders

app = FastAPI(title="OptiForge3D API")

# Configure torch settings at worker startup (expandable segments, etc.)
gpu_config.configure_torch()

class GenerationRequest(BaseModel):
    prompt: str
    guidance_scale: float = 15.0
    steps: int = 64

@app.get("/")
def read_root():
    return {
        "message": "OptiForge3D API is ready!",
        "models_loaded_in_cache": os.listdir('./models') if os.path.exists('./models') else []
    }

@app.post("/generate")
def generate_3d(req: GenerationRequest):
    \"\"\"
    Endpoint that generates a 3D model using Shap-E and returns the .ply file.
    \"\"\"
    try:
        from shap_e.diffusion.sample import sample_latents
        from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
        from shap_e.models.download import load_config
        from shap_e.util.notebooks import decode_latent_mesh

        # Load Shap-E models
        xm, _ = loaders.load_shap_e("transmitter")
        shap_e_model, device = loaders.load_shap_e("text300M")
        diffusion = diffusion_from_config(load_config('diffusion'))

        gpu_config.log_gpu_memory("after Shap-E load")

        # Generate latents
        latents = sample_latents(
            batch_size=1,
            model=shap_e_model,
            diffusion=diffusion,
            guidance_scale=req.guidance_scale,
            model_kwargs=dict(texts=[req.prompt]),
            progress=False,
            clip_denoised=True,
            use_fp16=True,
            use_karras=True,
            karras_steps=req.steps,
            sigma_min=1e-3,
            sigma_max=160,
            s_churn=0,
        )

        # Decode mesh
        mesh = decode_latent_mesh(xm, latents[0])

        # Save to disk
        output_file = "output_mesh.obj"
        with open(output_file, "w") as f:
            mesh.tri_mesh().write_obj(f)

        # Unload models
        del shap_e_model, xm, diffusion, latents, mesh
        gpu_config.free_gpu_memory()

        return FileResponse(output_file, media_type="application/octet-stream", filename="generated_mesh.obj")

    except Exception as e:
        return {"status": "error", "message": str(e)}

class YoloRequest(BaseModel):
    image_url: str

@app.post("/test-yolov8")
def test_yolo(req: YoloRequest):
    try:
        import requests
        from PIL import Image
        from io import BytesIO

        # Load YOLOv8 model
        yolo_model = loaders.load_yolov8()
        gpu_config.log_gpu_memory("after YOLOv8 load")

        # Download and process image
        img_resp = requests.get(req.image_url)
        img = Image.open(BytesIO(img_resp.content))

        results = yolo_model(img)
        detections = [
            {"class": yolo_model.names[int(box.cls)], "confidence": float(box.conf)}
            for box in results[0].boxes
        ]

        # Unload
        del yolo_model
        gpu_config.free_gpu_memory()

        return {"status": "success", "detections": detections}
    except Exception as e:
        return {"status": "error", "message": str(e)}

class ClipRequest(BaseModel):
    prompt: str

@app.post("/test-clip")
def test_clip(req: ClipRequest):
    try:
        # Load CLIP model
        clip_model, clip_processor, device = loaders.load_clip()
        gpu_config.log_gpu_memory("after CLIP load")

        # Run text processing
        with torch.no_grad():
            inputs = clip_processor(text=[req.prompt], return_tensors="pt").to(device)
            features = clip_model.get_text_features(**inputs)

        shape = list(features.shape) if hasattr(features, "shape") else "Successfully generated embeddings"


        # Unload
        del clip_model, clip_processor, inputs, features
        gpu_config.free_gpu_memory()

        return {"status": "success", "embedding_shape": shape}
    except Exception as e:
        return {"status": "error", "message": str(e)}
"""
with open("app.py", "w") as f:
    f.write(api_script)


## 5. Start Server & Tunnel
Spins up the optimized FastAPI server and a secure Ngrok tunnel (no timeouts, no passwords).


In [5]:
import threading
import time
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

def run_server():
    uvicorn.run("app:app", host="127.0.0.1", port=8000)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# IMPORTANT: Put your free Ngrok auth token here!
# Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "2ys7XxnIEUl6tZc0p4Tqtw5Xesu_7ZpzzZUUARcHJ85QnYvUD"

print("Starting Ngrok Tunnel...")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url
print(f"Your API is live at: {public_url}")


Starting Ngrok Tunnel...


INFO:     Started server process [6464]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Your API is live at: https://d2a8-35-240-180-120.ngrok-free.app
